# ProductScope — Task 04 Google Colab
## Extract product names, prices and ratings and save them as CSV

Use only public pages where automated collection is permitted.

In [ ]:
!pip -q install requests beautifulsoup4 pandas

import requests, json, re, pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse


In [ ]:
URL = 'https://example.com/products'  # Replace with a permitted public product/category URL

response = requests.get(
    URL,
    timeout=20,
    headers={'User-Agent': 'Mozilla/5.0 (Educational ProductScope Project)'}
)
response.raise_for_status()
soup = BeautifulSoup(response.text, 'html.parser')
print('Fetched:', urlparse(URL).netloc)


In [ ]:
def clean_text(value):
    return re.sub(r'\s+', ' ', str(value or '')).strip()

def parse_number(value):
    match = re.search(r'[-+]?\d*\.?\d+', clean_text(value).replace(',', ''))
    return float(match.group()) if match else None

def walk_json(value):
    if isinstance(value, dict):
        yield value
        for child in value.values():
            yield from walk_json(child)
    elif isinstance(value, list):
        for child in value:
            yield from walk_json(child)


In [ ]:
products = []

# Preferred method: structured JSON-LD Product data
for script in soup.select('script[type="application/ld+json"]'):
    try:
        data = json.loads(script.get_text(strip=True))
    except Exception:
        continue
    for item in walk_json(data):
        item_type = item.get('@type', '')
        types = item_type if isinstance(item_type, list) else [item_type]
        if 'Product' not in types:
            continue
        offers = item.get('offers', {})
        if isinstance(offers, list):
            offers = offers[0] if offers else {}
        rating_data = item.get('aggregateRating', {})
        products.append({
            'name': clean_text(item.get('name')),
            'price': parse_number(offers.get('price') or item.get('price')),
            'rating': parse_number(rating_data.get('ratingValue') if isinstance(rating_data, dict) else None),
            'currency': clean_text(offers.get('priceCurrency')) if isinstance(offers, dict) else '',
            'url': urljoin(URL, item.get('url') or URL),
            'method': 'JSON-LD'
        })

# Fallback method: common HTML product cards
if not products:
    cards = soup.select('article, .product-card, .product-item, .product, [data-product], [itemtype*="Product"]')
    for card in cards[:200]:
        def first(selectors):
            for selector in selectors:
                node = card.select_one(selector)
                if node:
                    value = node.get('content') or node.get('data-price') or node.get_text(' ', strip=True)
                    if clean_text(value): return clean_text(value)
            return ''
        name = first(['[itemprop="name"]','.product-title','.product-name','.title','h1','h2','h3','h4'])
        price = first(['[itemprop="price"]','.product-price','.price','[data-price]','[class*="price"]'])
        rating = first(['[itemprop="ratingValue"]','.rating','.stars','[class*="rating"]'])
        link = card.select_one('a[href]')
        if name:
            products.append({'name':name,'price':parse_number(price),'rating':parse_number(rating),'currency':'','url':urljoin(URL,link['href']) if link else URL,'method':'HTML'})

# Remove duplicate names
seen = set(); clean_products = []
for product in products:
    key = product['name'].casefold()
    if product['name'] and key not in seen:
        seen.add(key); clean_products.append(product)

df = pd.DataFrame(clean_products)
df


In [ ]:
df.to_csv('productscope_products.csv', index=False)
print(f'Saved {len(df)} records to productscope_products.csv')

from google.colab import files
files.download('productscope_products.csv')
